# 📚 ДЗ №2: Работа с данными для LLM

## 🎯 Цель задания
После выполнения задания вы сможете:
- Предобрабатывать русскоязычные текстовые данные для LLM
- Работать с готовыми моделями HuggingFace для анализа тональности и NER
- Создавать эффективные промпты для LLM API
- Сравнивать качество работы разных подходов к анализу текста
- Формировать датасеты в формате instruction-following для fine-tuning
- Сохранять данные в правильных форматах для обучения LLM

## 📝 Структура задания
- **Часть 1** (35% оценки): Предобработка данных и работа с готовыми моделями
- **Часть 2** (35% оценки): LLM API и prompt engineering
- **Часть 3** (20% оценки): Подготовка данных для fine-tuning LLM
- **Часть 4** (10% оценки): Сравнительный анализ и визуализация

## ⚡ Критерии оценки
- Качество предобработки данных: 25%
- Корректность работы с готовыми моделями: 20%
- Эффективность промптов для LLM: 25%
- Правильность подготовки данных для fine-tuning: 20%
- Качество сравнительного анализа: 10%


## 🔧 Установка зависимостей

Установим необходимые библиотеки для работы с данными, готовыми моделями и LLM API.


In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn
%pip install transformers torch
%pip install openai>=1.0.0  # Для работы с OpenAI API
%pip install datasets huggingface_hub python-dotenv
%pip install pymorphy2

In [ ]:
# Импорт необходимых библиотек
import os
import json
import re
import typing

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings

from transformers import AutoModelForTokenClassification, AutoTokenizer, pipeline
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# Загружаем секреты из .env (файл не коммитится в git).
# В Colab можно вместо .env положить токен в Secrets и сделать:
#   os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    print("⚠️  HF_TOKEN не найден. Создайте файл .env (см. .env.example) "
          "или задайте переменную окружения HF_TOKEN.")

# Настройка отображения
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("Библиотеки загружены успешно!")

## 📊 Часть 1: Предобработка данных и готовые модели (35% оценки)

### Задание 1.1: Анализ "грязного" датасета

Проанализируем реалистичный датасет с типичными проблемами: опечатки, разные регистры, лишние пробелы, эмодзи.


In [4]:
# Создаем "грязный" датасет с типичными проблемами реальных данных
# Включаем сложные случаи для демонстрации преимуществ LLM
raw_reviews = [
    # Простые случаи
    "отличный iphone 14 PRO!!!  купил в магазине  apple на тверской 😊. Камера супер",
    "УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(",

    # Сарказм и ирония (сложно для классических моделей)
    "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏",
    "Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился",

    # Смешанные эмоции
    "iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store",
    "Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен",

    # Сложная структура предложений
    "Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой",
    "Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги",

    # Контекстно-зависимые случаи
    "Заказал доставку в Яндекс.Еде из ресторана Дача на Рублевке - привезли холодное, но курьер Андрей был вежливый",
    "MacBook Pro 16 работает как часы уже год, покупал в iStore на Арбате у консультанта Елены",

    # Неоднозначные случаи
    "Сходил в кинотеатр Октябрь посмотреть новый фильм Marvel - ну такое себе, но попкорн вкусный был",
    "Обслуживание в банке ВТБ на Тверской оставляет желать лучшего, хотя менеджер Ольга старалась помочь",

    # Сложные именованные сущности
    "Купил новый Samsung Galaxy S24 Ultra в DNS на Ленинском проспекте, консультант Дмитрий Иванович всё объяснил",
    "Ужинал в ресторане White Rabbit на Смоленской площади - шеф-повар Владимир Мухин превзошел ожидания",

    # Опечатки и сленг
    "норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции"
]

# TODO: Создайте DataFrame и проанализируйте проблемы в данных
# Создайте DataFrame из списка raw_reviews
# Добавьте колонку с правильными метками тональности для каждого отзыва
# Проанализируйте и выведите список проблем, которые вы видите в данных
# Подумайте: какие проблемы могут повлиять на качество анализа?

# Ваш код здесь:
import pandas as pd

# Правильные метки тональности (Ground Truth)
sentiments = [
    "positive", "negative", "negative", "negative", "positive", 
    "neutral", "positive", "neutral", "neutral", "positive", 
    "neutral", "negative", "positive", "positive", "positive"
]
# Создаем DataFrame - где поля это колонки
df = pd.DataFrame({
    'review': raw_reviews,
    'true_sentiment': sentiments
})

print(df.head())


# Регистр: Смешивание CAPS LOCK ("УЖАСНОЕ") и строчных букв. Для многих моделей это разные токены
# Пунктуация: Избыточные знаки ("!!!", "..") и использование скобок в качестве смайликов ("("), что может сбивать токенизаторы
# Эмодзи: Наличие графических символов (😊, 👏). Они несут смысл, но требуют специальной обработки
# Сарказм: Отзывы про МТС и Пятерочку содержат позитивные слова ("спасибо", "восхитительно", "замечательный")
# Смешанные чувства (Mixed Sentiment): В отзыве про «Яндекс.Еду» есть и минус (холодное), и плюс (вежливый курьер). Это усложняет задачу классификации до одного класса
# Опечатки и сленг: "вобще", "норм", "чел", "телек". Это требует нормализации или использования моделей, устойчивых к ошибкам


                                              review true_sentiment
0  отличный iphone 14 PRO!!!  купил в магазине  a...       positive
1  УЖАСНОЕ обслуживание в сбербанке на красной пл...       negative
2  Спасибо огромное сотрудникам МТС за то что 3 ч...       negative
3  Какой замечательный сервис в Пятерочке - касса...       negative
4  iPhone 13 хороший телефон, но цена кусается. В...       positive


### Задание 1.2: Очистка и нормализация данных


In [ ]:
import re

def clean_text(text: str) -> str:
    """
    Очистка и нормализация русскоязычного текста ДЛЯ АНАЛИЗА ТОНАЛЬНОСТИ.
    Приводим к нижнему регистру: для sentiment-моделей регистр не важен,
    а нормализация уменьшает число "разных" токенов для одного слова.
    """
    # 1. Исправление регистра (приводим к нижнему)
    text = text.lower()

    # 2. Разделение слитно написанных слов (буквы и цифры: iPhone14 -> iphone 14)
    text = re.sub(r'([a-zа-яё])(\d)', r'\1 \2', text)
    text = re.sub(r'(\d)([a-zа-яё])', r'\1 \2', text)

    # 3. Обработка повторяющихся знаков препинания (!!! -> !, .. -> .)
    text = re.sub(r'([!?.]){2,}', r'\1', text)

    # 4. Удаление эмодзи и "странных" спецсимволов
    text = re.sub(r'[^а-яёa-z0-9\s\.,!?]', ' ', text)

    # 5. Нормализация пробелов и отступов
    text = re.sub(r'\s+', ' ', text).strip()

    return text


def clean_text_for_ner(text: str) -> str:
    """
    Лёгкая очистка ДЛЯ NER. В отличие от clean_text здесь мы СОХРАНЯЕМ
    регистр: заглавные буквы критичны для распознавания имён собственных
    (Иван, Сбербанк, Tesla). Если всё привести к нижнему регистру, NER-модель
    перестаёт отличать имена от обычных слов — именно поэтому в исходном
    варианте сущности находились плохо.
    """
    # Убираем эмодзи/графические символы, но НЕ трогаем регистр и пунктуацию.
    # \w с флагом UNICODE сохраняет кириллицу и латиницу.
    text = re.sub(r'[^\w\s\.,!?:\-]', ' ', text, flags=re.UNICODE)
    # Чиним повторяющиеся знаки препинания
    text = re.sub(r'([!?.]){2,}', r'\1', text)
    # Нормализуем пробелы
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Применяем обе функции: одну для тональности, другую для NER
df['cleaned_review'] = df['review'].apply(clean_text)
df['cleaned_for_ner'] = df['review'].apply(clean_text_for_ner)

# Выведем сравнение для нескольких случаев
pd.set_option('display.max_colwidth', None)  # Чтобы текст не обрезался при выводе
comparison = df[['review', 'cleaned_review', 'cleaned_for_ner']].head(10)
print(comparison)

### Задание 1.3: Использование готовых моделей HuggingFace


In [ ]:
from transformers import pipeline

# Токен берём из переменной окружения HF_TOKEN (загружена из .env в ячейке с импортами)

# Модель для анализа тональности (русскоязычная, 3 класса: NEGATIVE/NEUTRAL/POSITIVE)
sentiment_model = pipeline(
    "sentiment-analysis",
    model="MonoHime/rubert-base-cased-sentiment-new",
    token=HF_TOKEN
)

# Модель для NER (мультиязычная, понимает русский и английский)
ner_model = pipeline(
    "ner",
    model="Babelscape/wikineural-multilingual-ner",
    aggregation_strategy="simple"
)


def analyze_with_huggingface(texts: list, ner_texts: list = None) -> list:
    """
    Анализ текстов готовыми моделями HuggingFace.

    Важно: для тональности используем нормализованный (lowercase) текст,
    а для NER — текст С СОХРАНЁННЫМ регистром (ner_texts), т.к. заглавные
    буквы критичны для распознавания имён собственных.
    """
    if ner_texts is None:
        ner_texts = texts

    results = []
    # pipeline умеет обрабатывать списки целиком — это быстрее, чем цикл
    sentiments = sentiment_model(texts)
    entities_list = ner_model(ner_texts)

    for i in range(len(texts)):
        results.append({
            "text": texts[i],
            "sentiment": sentiments[i]['label'],
            "sentiment_score": round(sentiments[i]['score'], 3),
            "entities": [
                {"word": ent['word'], "type": ent['entity_group']}
                for ent in entities_list[i]
            ]
        })

    return results


# Тестируем на первых 5 отзывах: тональность по cleaned_review, NER по cleaned_for_ner
test_df = df.head(5)
analysis_results = analyze_with_huggingface(
    test_df['cleaned_review'].tolist(),
    test_df['cleaned_for_ner'].tolist()
)

for res in analysis_results:
    print(f"Текст: {res['text']}")
    print(f"Тональность: {res['sentiment']} (уверенность: {res['sentiment_score']})")
    print(f"Сущности: {res['entities']}")
    print("-" * 50)

### 📌 Выводы по Части 1

- **Очистка зависит от задачи.** Для тональности нормализуем регистр (`clean_text`), а для NER регистр **сохраняем** (`clean_text_for_ner`) — иначе модель перестаёт отличать имена собственные от обычных слов.
- **Sentiment-модель** `MonoHime/rubert-base-cased-sentiment-new` хорошо ловит явную тональность, но ошибается на сарказме (отзыв про МТС → POSITIVE вместо NEGATIVE).
- **NER-модель** `Babelscape/wikineural-multilingual-ner` на тексте с сохранённым регистром находит бренды и имена заметно лучше, чем на полностью строчном тексте.

## 🤖 Часть 2: LLM API и Prompt Engineering (35% оценки)

### Задание 2.1: Создание эффективных промптов


In [ ]:
def create_prompts_for_llm() -> dict:
    """
    Создание базовых промптов для разных задач (один промпт на задачу)
    """
    # Структура хорошего промпта: чёткая роль, описание задачи, перечень
    # допустимых ответов, строгий формат вывода (JSON) и учёт специфики
    # русского языка (сарказм, ирония).

    # Промпт для тональности. Явно требуем учитывать сарказм — это критично
    # для нашего датасета.
    sentiment_prompt = """
    Ты — эксперт по лингвистическому анализу. Твоя задача — определить тональность текста на русском языке.
    Учти, что текст может содержать сарказм или скрытую иронию.

    Варианты ответа: POSITIVE, NEGATIVE, NEUTRAL.

    Верни ответ строго в формате JSON:
    {"sentiment": "категория", "confidence": 0.95, "reasoning": "краткое пояснение почему"}
    """

    # Промпт для NER
    ner_prompt = """
    Извлеки из текста все именованные сущности на русском или английском языках.
    Типы сущностей: PER (человек), ORG (организация/бренд), LOC (местоположение), PRODUCT (товар).

    Верни ответ строго в формате JSON:
    {"entities": [{"text": "название", "type": "тип"}]}
    """

    return {
        "sentiment": sentiment_prompt,
        "ner": ner_prompt
    }


import os
import json
from huggingface_hub import InferenceClient

# Настройка HuggingFace Inference Client.
# Токен берём из переменной окружения HF_TOKEN (загружена из .env).
client_hf = InferenceClient(token=HF_TOKEN)


def call_llm(system_prompt: str, user_text: str):
    """
    Вызов LLM по API с обработкой ошибок и парсингом JSON-ответа.
    Параметры: temperature=0.1 (детерминированность), max_tokens=500.
    """
    try:
        # У InferenceClient метод называется chat_completion (БЕЗ 's' и БЕЗ 'create')
        response = client_hf.chat_completion(
            model="meta-llama/Meta-Llama-3-8B-Instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text}
            ],
            max_tokens=500,
            temperature=0.1
        )

        # Извлекаем текст ответа
        content = response.choices[0].message.content

        # Очистка ответа от Markdown-разметки JSON (модель иногда оборачивает в ```)
        clean_response = content.strip()
        if "```json" in clean_response:
            clean_response = clean_response.split("```json")[1].split("```")[0]
        elif "```" in clean_response:
            clean_response = clean_response.split("```")[1].split("```")[0]

        return json.loads(clean_response.strip())

    except Exception as e:
        print(f"Ошибка вызова: {e}")
        return None


# ТЕСТ
prompts = create_prompts_for_llm()
test_text = "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏"
print("Результат LLM:", call_llm(prompts["sentiment"], test_text))

### Задание 2.2: Сравнение результатов HuggingFace vs LLM


In [ ]:
# Сравнение результатов HuggingFace моделей с LLM на одних и тех же текстах.
# 1. Собираем результаты обоих подходов
# 2. Сравниваем точность тональности
# 3. Сравниваем извлечение сущностей
# 4. Замеряем время выполнения

import pandas as pd
import numpy as np
import time
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 70)
print("СРАВНЕНИЕ HUGGINGFACE vs LLM")
print("=" * 70)

# Шаг 1: Анализ с HuggingFace моделями
print("\n[1/4] Анализ с HuggingFace моделями...")
hf_start_time = time.time()

# Тональность — по нормализованному тексту
hf_sentiments = sentiment_model(df['cleaned_review'].tolist())
# NER — по тексту с сохранённым регистром (так модель находит имена собственные)
hf_ner_results = ner_model(df['cleaned_for_ner'].tolist())

hf_time = time.time() - hf_start_time

df['hf_sentiment'] = [result['label'].lower() for result in hf_sentiments]
df['hf_sentiment_score'] = [result['score'] for result in hf_sentiments]
df['hf_entities'] = hf_ner_results
df['hf_entities_count'] = df['hf_entities'].apply(len)

print(f"✓ HuggingFace завершён за {hf_time:.2f} сек")

# Шаг 2: Анализ с LLM
print("\n[2/4] Анализ с LLM (Llama-3)...")
llm_start_time = time.time()

prompts = create_prompts_for_llm()

llm_results = []
for text in df['cleaned_review']:
    sentiment_result = call_llm(prompts["sentiment"], text)
    ner_result = call_llm(prompts["ner"], text)

    llm_results.append({
        'sentiment': sentiment_result.get('sentiment', 'NEUTRAL').lower() if sentiment_result else 'neutral',
        'confidence': sentiment_result.get('confidence', 0.0) if sentiment_result else 0.0,
        'entities': ner_result.get('entities', []) if ner_result else []
    })

    # Небольшая задержка, чтобы не упереться в rate limit
    time.sleep(0.5)

llm_time = time.time() - llm_start_time

df['llm_sentiment'] = [r['sentiment'] for r in llm_results]
df['llm_sentiment_score'] = [r['confidence'] for r in llm_results]
df['llm_entities'] = [r['entities'] for r in llm_results]
df['llm_entities_count'] = df['llm_entities'].apply(len)

print(f"✓ LLM завершён за {llm_time:.2f} сек")

# Шаг 3: Сравнение точности
print("\n[3/4] Расчёт метрик точности...")

label_map = {
    'positive': 'positive', 'negative': 'negative', 'neutral': 'neutral',
    'pos': 'positive', 'neg': 'negative', 'neu': 'neutral'
}

df['hf_sentiment_normalized'] = df['hf_sentiment'].map(lambda x: label_map.get(x.lower(), x.lower()))
df['llm_sentiment_normalized'] = df['llm_sentiment'].map(lambda x: label_map.get(x.lower(), x.lower()))

acc_hf = accuracy_score(df['true_sentiment'], df['hf_sentiment_normalized'])
acc_llm = accuracy_score(df['true_sentiment'], df['llm_sentiment_normalized'])

print(f"\n📊 МЕТРИКИ ТОЧНОСТИ:")
print(f"   HuggingFace Accuracy: {acc_hf:.2%}")
print(f"   LLM Accuracy:         {acc_llm:.2%}")
print(f"   Разница:              {abs(acc_llm - acc_hf):.2%}")

print(f"\n📋 Classification Report - HuggingFace:")
print(classification_report(df['true_sentiment'], df['hf_sentiment_normalized'], zero_division=0))

print(f"\n📋 Classification Report - LLM:")
print(classification_report(df['true_sentiment'], df['llm_sentiment_normalized'], zero_division=0))

# Шаг 4: Визуализация
print("\n[4/4] Создание визуализаций...")

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# График 1: Сравнение точности
ax1 = axes[0, 0]
models = ['HuggingFace', 'LLM']
accuracies = [acc_hf, acc_llm]
colors = ['#3498db', '#e74c3c']
bars = ax1.bar(models, accuracies, color=colors, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Accuracy', fontsize=12)
ax1.set_title('Сравнение точности моделей', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 1)
ax1.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.2%}', ha='center', fontsize=11, fontweight='bold')

# График 2: Время выполнения
ax2 = axes[0, 1]
times = [hf_time, llm_time]
bars = ax2.bar(models, times, color=colors, alpha=0.7, edgecolor='black')
ax2.set_ylabel('Время (секунды)', fontsize=12)
ax2.set_title('Время обработки всех текстов', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)
for bar, t in zip(bars, times):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{t:.1f}s', ha='center', fontsize=11, fontweight='bold')

# График 3: Количество найденных сущностей
ax3 = axes[1, 0]
avg_entities_hf = df['hf_entities_count'].mean()
avg_entities_llm = df['llm_entities_count'].mean()
entity_counts = [avg_entities_hf, avg_entities_llm]
bars = ax3.bar(models, entity_counts, color=colors, alpha=0.7, edgecolor='black')
ax3.set_ylabel('Среднее количество', fontsize=12)
ax3.set_title('Среднее количество найденных сущностей', fontsize=14, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)
for bar, count in zip(bars, entity_counts):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{count:.1f}', ha='center', fontsize=11, fontweight='bold')

# График 4: Confusion Matrix для LLM
ax4 = axes[1, 1]
cm = confusion_matrix(df['true_sentiment'], df['llm_sentiment_normalized'],
                       labels=['positive', 'negative', 'neutral'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=['positive', 'negative', 'neutral'],
            yticklabels=['positive', 'negative', 'neutral'])
ax4.set_ylabel('True Label', fontsize=12)
ax4.set_xlabel('Predicted Label', fontsize=12)
ax4.set_title('Confusion Matrix - LLM', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Итоговое сравнение
print("\n" + "=" * 70)
print("📊 ИТОГОВОЕ СРАВНЕНИЕ")
print("=" * 70)

comparison_summary = pd.DataFrame({
    'Метрика': ['Accuracy', 'Время (сек)', 'Avg Entities', 'Цена за вызов'],
    'HuggingFace': [f"{acc_hf:.2%}", f"{hf_time:.2f}", f"{avg_entities_hf:.1f}", "Бесплатно"],
    'LLM': [f"{acc_llm:.2%}", f"{llm_time:.2f}", f"{avg_entities_llm:.1f}", "~$0.001"],
})

print(comparison_summary.to_string(index=False))

print("\n💡 ВЫВОДЫ:")
print("   1. Точность: LLM обычно лучше понимает сарказм и контекст")
print("   2. Скорость: HuggingFace быстрее, т.к. работает локально")
print("   3. Стоимость: HuggingFace бесплатен, LLM требует API токенов")
print("   4. NER: Зависит от модели, но LLM часто более гибкий")
print("   5. Рекомендация: HF для массовой обработки, LLM для сложных случаев")
print("=" * 70)

### Задания 2.3–2.5: качественный и количественный анализ

Используем предсказания обеих моделей, посчитанные в задании 2.2 (хранятся в `df`): качественный разбор сложных случаев, метрики по классам с анализом ошибок и визуализация.

In [ ]:
# Задание 2.3: Качественный анализ сложных случаев (HuggingFace vs LLM)
#
# Берём из общего df (он уже содержит предсказания обеих моделей, посчитанные
# в задании 2.2) самые сложные для классических моделей примеры:
# сарказм, смешанные эмоции, сложную структуру и сленг.

complex_idx = [2, 4, 6, 14]  # МТС(сарказм), iPhone13(смеш.), Tesla(сложн.), LG(сленг)

print("Сравнение на сложных случаях:")
print("=" * 90)
for i in complex_idx:
    row = df.iloc[i]
    hf_ok = "✓" if row['hf_sentiment_normalized'] == row['true_sentiment'] else "✗"
    llm_ok = "✓" if row['llm_sentiment_normalized'] == row['true_sentiment'] else "✗"
    print(f"\nТекст: {row['review']}")
    print(f"  Эталон (true): {row['true_sentiment']}")
    print(f"  HuggingFace:   {row['hf_sentiment_normalized']:<9} (conf={row['hf_sentiment_score']:.2f})  {hf_ok}")
    print(f"  LLM (Llama-3): {row['llm_sentiment_normalized']:<9} (conf={row['llm_sentiment_score']:.2f})  {llm_ok}")
print("\n" + "=" * 90)
print("Вывод: на сарказме ('спасибо, что держали в очереди') и смешанных эмоциях")
print("LLM, как правило, точнее — она оценивает смысл фразы целиком, а классическая")
print("модель ориентируется на отдельные 'позитивные'/'негативные' слова и ошибается.")

In [ ]:
# Задание 2.4: Количественное сравнение точности (по классам + анализ ошибок)

from sklearn.metrics import classification_report, accuracy_score

labels = ['positive', 'negative', 'neutral']

rep_hf = classification_report(df['true_sentiment'], df['hf_sentiment_normalized'],
                               labels=labels, output_dict=True, zero_division=0)
rep_llm = classification_report(df['true_sentiment'], df['llm_sentiment_normalized'],
                                labels=labels, output_dict=True, zero_division=0)

# F1-score по каждому классу — показывает, на каких классах модель сильнее
f1_table = pd.DataFrame({
    'Класс': labels,
    'F1 (HuggingFace)': [round(rep_hf[c]['f1-score'], 2) for c in labels],
    'F1 (LLM)': [round(rep_llm[c]['f1-score'], 2) for c in labels],
})
print("F1-score по классам:")
print(f1_table.to_string(index=False))

print(f"\nОбщая точность (accuracy):")
print(f"  HuggingFace: {accuracy_score(df['true_sentiment'], df['hf_sentiment_normalized']):.2%}")
print(f"  LLM:         {accuracy_score(df['true_sentiment'], df['llm_sentiment_normalized']):.2%}")

# Анализ ошибок: на каких именно отзывах модели разошлись с эталоном
print("\n" + "-" * 70)
print("Ошибки HuggingFace:")
err_hf = df[df['hf_sentiment_normalized'] != df['true_sentiment']]
for _, r in err_hf.iterrows():
    print(f"  '{r['review'][:55]}...'\n     → предсказано: {r['hf_sentiment_normalized']}, верно: {r['true_sentiment']}")

print("\nОшибки LLM:")
err_llm = df[df['llm_sentiment_normalized'] != df['true_sentiment']]
for _, r in err_llm.iterrows():
    print(f"  '{r['review'][:55]}...'\n     → предсказано: {r['llm_sentiment_normalized']}, верно: {r['true_sentiment']}")

print("\n" + "-" * 70)
print(f"Всего ошибок — HuggingFace: {len(err_hf)}, LLM: {len(err_llm)} (из {len(df)} отзывов)")

In [ ]:
# Задание 2.5: Визуализация сравнения моделей (F1 по классам + согласованность)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

labels = ['positive', 'negative', 'neutral']
rep_hf = classification_report(df['true_sentiment'], df['hf_sentiment_normalized'],
                               labels=labels, output_dict=True, zero_division=0)
rep_llm = classification_report(df['true_sentiment'], df['llm_sentiment_normalized'],
                                labels=labels, output_dict=True, zero_division=0)

f1_hf = [rep_hf[c]['f1-score'] for c in labels]
f1_llm = [rep_llm[c]['f1-score'] for c in labels]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, f1_hf, w, label='HuggingFace', color='#3498db', alpha=0.8, edgecolor='black')
b2 = ax.bar(x + w/2, f1_llm, w, label='LLM (Llama-3)', color='#e74c3c', alpha=0.8, edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('F1-score')
ax.set_ylim(0, 1.05)
ax.set_title('F1-score по классам: HuggingFace vs LLM', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
                f'{b.get_height():.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# Насколько часто модели согласны между собой (вне зависимости от эталона)
agree = (df['hf_sentiment_normalized'] == df['llm_sentiment_normalized']).mean()
print(f"Согласованность HF и LLM между собой: {agree:.0%}")
print("Низкая согласованность на сложных классах (neutral/сарказм) показывает,")
print("что подходы по-разному трактуют неоднозначные отзывы.")

### 📌 Выводы по Части 2

- **LLM точнее на сарказме и сложных формулировках** за счёт понимания контекста, тогда как HF-модель ориентируется на отдельные слова.
- **HF быстрее и бесплатна**, но «спотыкается» на иронии и смешанных эмоциях.
- **Хороший промпт** = чёткая роль + перечень допустимых ответов + строгий JSON-формат + явное требование учитывать сарказм. Это даёт стабильный парсируемый ответ.
- **Рекомендация:** HF — для массовой дешёвой обработки, LLM — для сложных/неоднозначных случаев.

## 📚 Часть 3: Подготовка данных для Fine-tuning LLM (20% оценки)

### Задание 3.1: Создание instruction-following датасета

In [6]:
def create_instruction_dataset(df: pd.DataFrame) -> list[dict]:
    """
    Создание датасета в формате instruction-following для fine-tuning LLM
    """
    # TODO: Создайте структурированный датасет для fine-tuning LLM
    # Подумайте о структуре instruction-following датасета:
    # - Какие поля должны быть в каждом примере?
    # - Как сформулировать инструкции для модели?
    # - Какие типы задач включить (sentiment, NER, etc.)?
    # - Как структурировать ответы модели?
    #
    # Создайте несколько примеров для разных задач

    instruction_data = []
    
    for idx, row in df.iterrows():
        # Задача 1: Анализ тональности (Sentiment Analysis)
        sentiment_example = {
            "instruction": "Определи тональность следующего отзыва. Ответь одним словом: positive, negative или neutral.",
            "input": row['cleaned_review'],
            "output": row['true_sentiment'],
            "task_type": "sentiment_analysis"
        }
        instruction_data.append(sentiment_example)
        
        # Задача 2: Анализ тональности с объяснением (для более сложного обучения)
        sentiment_detailed = {
            "instruction": "Проанализируй тональность отзыва и объясни свой вывод. Учитывай сарказм и иронию.",
            "input": row['cleaned_review'],
            "output": f"Тональность: {row['true_sentiment']}. Обоснование: текст содержит {'позитивные' if row['true_sentiment'] == 'positive' else 'негативные' if row['true_sentiment'] == 'negative' else 'нейтральные'} индикаторы.",
            "task_type": "sentiment_analysis_detailed"
        }
        instruction_data.append(sentiment_detailed)
        
        # Задача 3: Извлечение именованных сущностей (NER)
        # Используем результаты LLM, если они есть
        if 'llm_entities' in df.columns and len(row['llm_entities']) > 0:
            entities_str = ", ".join([f"{e['text']} ({e['type']})" for e in row['llm_entities']])
            ner_example = {
                "instruction": "Извлеки все именованные сущности из текста: организации, люди, места, продукты. Формат: название (тип).",
                "input": row['cleaned_review'],
                "output": entities_str,
                "task_type": "ner"
            }
            instruction_data.append(ner_example)
        
        # Задача 4: Классификация по категориям (Multi-task)
        multi_task = {
            "instruction": "Выполни комплексный анализ отзыва: определи тональность и извлеки ключевые объекты (бренды, продукты, места).",
            "input": row['cleaned_review'],
            "output": f"Тональность: {row['true_sentiment']}. Объекты: {row.get('cleaned_review', '')}",
            "task_type": "multi_task"
        }
        instruction_data.append(multi_task)
    
    return instruction_data


# TODO: Протестируйте созданный датасет
# Создайте и проанализируйте instruction dataset
# Выведите примеры в читаемом формате
# Проанализируйте распределение типов задач

print("=" * 70)
print("СОЗДАНИЕ INSTRUCTION-FOLLOWING ДАТАСЕТА")
print("=" * 70)

# Создаем датасет
instruction_dataset = create_instruction_dataset(df)

print(f"\n📊 Статистика датасета:")
print(f"   Всего примеров: {len(instruction_dataset)}")
print(f"   Уникальных отзывов: {len(df)}")
print(f"   Примеров на отзыв: {len(instruction_dataset) // len(df)}")

# Анализ распределения типов задач
task_types = {}
for item in instruction_dataset:
    task_type = item['task_type']
    task_types[task_type] = task_types.get(task_type, 0) + 1

print(f"\n📈 Распределение по типам задач:")
for task, count in task_types.items():
    print(f"   {task}: {count} примеров ({count/len(instruction_dataset)*100:.1f}%)")

# Выводим несколько примеров
print(f"\n📝 Примеры из датасета:\n")
print("=" * 70)

# Выбираем разные типы задач для демонстрации
sample_tasks = ['sentiment_analysis', 'sentiment_analysis_detailed', 'ner', 'multi_task']
shown = set()

for task_type in sample_tasks:
    for item in instruction_dataset:
        if item['task_type'] == task_type and item['task_type'] not in shown:
            print(f"\n🔸 Тип задачи: {item['task_type']}")
            print(f"Инструкция: {item['instruction']}")
            print(f"Вход: {item['input'][:80]}..." if len(item['input']) > 80 else f"Вход: {item['input']}")
            print(f"Ожидаемый выход: {item['output'][:100]}..." if len(item['output']) > 100 else f"Ожидаемый выход: {item['output']}")
            print("-" * 70)
            shown.add(item['task_type'])
            break

# Создаем DataFrame для анализа
instruction_df = pd.DataFrame(instruction_dataset)

print(f"\n✅ Датасет успешно создан!")
print(f"   Готов для fine-tuning моделей типа GPT, Llama, и др.")

СОЗДАНИЕ INSTRUCTION-FOLLOWING ДАТАСЕТА

📊 Статистика датасета:
   Всего примеров: 45
   Уникальных отзывов: 15
   Примеров на отзыв: 3

📈 Распределение по типам задач:
   sentiment_analysis: 15 примеров (33.3%)
   sentiment_analysis_detailed: 15 примеров (33.3%)
   multi_task: 15 примеров (33.3%)

📝 Примеры из датасета:


🔸 Тип задачи: sentiment_analysis
Инструкция: Определи тональность следующего отзыва. Ответь одним словом: positive, negative или neutral.
Вход: отличный iphone 14 pro! купил в магазине apple на тверской . камера супер
Ожидаемый выход: positive
----------------------------------------------------------------------

🔸 Тип задачи: sentiment_analysis_detailed
Инструкция: Проанализируй тональность отзыва и объясни свой вывод. Учитывай сарказм и иронию.
Вход: отличный iphone 14 pro! купил в магазине apple на тверской . камера супер
Ожидаемый выход: Тональность: positive. Обоснование: текст содержит позитивные индикаторы.
----------------------------------------------------

### Задание 3.2: Сериализация данных в формате для LLM платформ


In [ ]:
# Сохранение данных в форматах для fine-tuning:
# 1. JSONL для OpenAI fine-tuning API
# 2. CSV для общего использования
# Имена файлов соответствуют требованиям задания:
#   fine_tuning_openai.jsonl и fine_tuning_data.csv

import json
import os

def save_to_openai_jsonl(data: list, filepath: str = "fine_tuning_openai.jsonl"):
    """
    Сохранение датасета в формате JSONL для OpenAI Fine-tuning API.

    Формат OpenAI требует:
    - messages: список с system, user, assistant
    - Каждая строка - отдельный JSON объект
    """
    print(f"\n📝 Сохранение в формат OpenAI JSONL...")

    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            # Формат для Chat Models (GPT-3.5/4)
            openai_format = {
                "messages": [
                    {
                        "role": "system",
                        "content": "Ты — эксперт по анализу русскоязычных отзывов. Следуй инструкциям точно."
                    },
                    {
                        "role": "user",
                        "content": f"{item['instruction']}\n\nТекст: {item['input']}"
                    },
                    {
                        "role": "assistant",
                        "content": item['output']
                    }
                ]
            }
            # Записываем каждый пример как отдельную строку JSON
            f.write(json.dumps(openai_format, ensure_ascii=False) + '\n')

    print(f"   ✓ Сохранено {len(data)} примеров в {filepath}")
    print(f"   ✓ Размер файла: {os.path.getsize(filepath) / 1024:.2f} KB")
    return filepath


def save_to_huggingface_jsonl(data: list, filepath: str = "finetune_hf.jsonl"):
    """
    Сохранение датасета в формате JSONL для HuggingFace / общего использования.

    Упрощенный формат с полями: instruction, input, output
    """
    print(f"\n📝 Сохранение в формат HuggingFace JSONL...")

    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            # Простой формат для HF datasets
            hf_format = {
                "instruction": item['instruction'],
                "input": item['input'],
                "output": item['output'],
                "task_type": item.get('task_type', 'unknown')
            }
            f.write(json.dumps(hf_format, ensure_ascii=False) + '\n')

    print(f"   ✓ Сохранено {len(data)} примеров в {filepath}")
    print(f"   ✓ Размер файла: {os.path.getsize(filepath) / 1024:.2f} KB")
    return filepath


def save_to_csv(data: list, filepath: str = "fine_tuning_data.csv"):
    """
    Сохранение датасета в формат CSV для общего использования.
    """
    print(f"\n📝 Сохранение в формат CSV...")

    df_export = pd.DataFrame(data)
    df_export.to_csv(filepath, index=False, encoding='utf-8')

    print(f"   ✓ Сохранено {len(data)} примеров в {filepath}")
    print(f"   ✓ Размер файла: {os.path.getsize(filepath) / 1024:.2f} KB")
    print(f"   ✓ Колонки: {', '.join(df_export.columns)}")
    return filepath


def load_and_validate_jsonl(filepath: str, format_type: str = "openai"):
    """
    Загрузка и валидация JSONL файла.
    """
    print(f"\n🔍 Валидация {filepath}...")

    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    print(f"   ✓ Загружено строк: {len(lines)}")

    # Проверяем первую запись
    first_item = json.loads(lines[0])
    print(f"   ✓ Структура первой записи:")

    if format_type == "openai":
        print(f"      - messages: {len(first_item.get('messages', []))} сообщений")
        for msg in first_item.get('messages', []):
            print(f"        • {msg['role']}: {msg['content'][:50]}...")
    else:
        print(f"      - instruction: {first_item.get('instruction', 'N/A')[:50]}...")
        print(f"      - input: {first_item.get('input', 'N/A')[:50]}...")
        print(f"      - output: {first_item.get('output', 'N/A')[:50]}...")

    return len(lines)


# ============================================================================
# ОСНОВНОЙ КОД: Сохранение датасета в разных форматах
# ============================================================================

print("=" * 70)
print("СЕРИАЛИЗАЦИЯ ДАННЫХ ДЛЯ FINE-TUNING")
print("=" * 70)

# Сохраняем в разных форматах (имена для OpenAI/CSV — как в задании)
openai_file = save_to_openai_jsonl(instruction_dataset, "fine_tuning_openai.jsonl")
hf_file = save_to_huggingface_jsonl(instruction_dataset, "finetune_hf.jsonl")
csv_file = save_to_csv(instruction_dataset, "fine_tuning_data.csv")

print("\n" + "=" * 70)
print("ВАЛИДАЦИЯ СОХРАНЕННЫХ ФАЙЛОВ")
print("=" * 70)

# Валидация файлов
openai_count = load_and_validate_jsonl(openai_file, format_type="openai")
hf_count = load_and_validate_jsonl(hf_file, format_type="hf")

# Загрузка CSV
print(f"\n🔍 Валидация {csv_file}...")
csv_df = pd.read_csv(csv_file)
print(f"   ✓ Загружено строк: {len(csv_df)}")
print(f"   ✓ Колонки: {', '.join(csv_df.columns)}")

print("\n" + "=" * 70)
print("📊 ИТОГОВАЯ СВОДКА")
print("=" * 70)

summary_df = pd.DataFrame({
    'Формат': ['OpenAI JSONL', 'HuggingFace JSONL', 'CSV'],
    'Файл': [openai_file, hf_file, csv_file],
    'Примеров': [openai_count, hf_count, len(csv_df)],
    'Размер (KB)': [
        f"{os.path.getsize(openai_file) / 1024:.2f}",
        f"{os.path.getsize(hf_file) / 1024:.2f}",
        f"{os.path.getsize(csv_file) / 1024:.2f}"
    ],
    'Назначение': [
        'OpenAI Fine-tuning',
        'HuggingFace / Общее',
        'Анализ данных'
    ]
})

print(summary_df.to_string(index=False))

print("\n💡 РЕКОМЕНДАЦИИ ПО ИСПОЛЬЗОВАНИЮ:")
print("   1. OpenAI JSONL: загрузите в OpenAI для fine-tuning GPT-3.5/4")
print("   2. HuggingFace JSONL: используйте с datasets.load_dataset()")
print("   3. CSV: для анализа и визуализации в pandas/Excel")
print("\n✅ Все файлы успешно сохранены и проверены!")
print("=" * 70)

## 📊 Часть 4: Итоговые выводы

**Точность.** LLM (Llama-3) сильнее на сложных классах — сарказм, ирония, смешанные эмоции, — потому что оценивает смысл фразы целиком. Классическая HF-модель надёжна на явной тональности, но ошибается на иронии (видит «позитивные» слова и ставит POSITIVE).

**Скорость.** HuggingFace работает локально и обрабатывает весь датасет за секунды; LLM по API заметно медленнее (сетевые задержки + rate limit), что видно на графике времени в задании 2.2.

**Простота и стоимость.** Готовая HF-модель — это один вызов `pipeline()` и она бесплатна. LLM требует токен, продуманный промпт и парсинг JSON-ответа, но зато гибко настраивается под задачу без дообучения.

**NER.** Ключевой фактор качества — сохранение регистра во входном тексте. После этого HF-NER находит бренды и имена адекватно; LLM при этом гибче в типах сущностей (PER/ORG/LOC/PRODUCT).

**Fine-tuning.** Собран instruction-following датасет (sentiment / sentiment+reasoning / multi-task) и сохранён в форматах `fine_tuning_openai.jsonl` (chat-формат для OpenAI) и `fine_tuning_data.csv`.

**Итоговая рекомендация:** HuggingFace — для массовой и дешёвой обработки, LLM — для сложных и неоднозначных случаев. Оптимально комбинировать: быстрый HF-фильтр + LLM на спорных примерах.